# FinBERT Fine-tune — Boilerplate Classifier (Stage 5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidavi111/bpclassifier/blob/main/notebooks/colab/finbert_finetune.ipynb)

**Runtime:** GPU (T4). Set via *Runtime → Change runtime type → T4 GPU*.

**What this does:**
1. Pulls `colab-data:v0` from W&B (train + val splits with sentence text)
2. Fine-tunes `ProsusAI/finbert` with 5-fold stratified group-aware CV
3. Trains a final model on the full train split
4. Pushes `model-finbert:v0` and `oof-finbert:v0` artifacts back to W&B

**Before running:** Add `WANDB_API_KEY` to Colab Secrets (key icon in left panel).

In [ ]:
!pip install -q wandb transformers accelerate pyarrow scikit-learn

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    print("Colab Secrets not found — prompting for login.")

import wandb
wandb.login()

WANDB_ENTITY  = "david-avichzer-hebrew-university-of-jerusalem"
WANDB_PROJECT = "Boilerplate_Classifier"
MODEL_NAME    = "ProsusAI/finbert"
N_FOLDS       = 5
SEED          = 42
EPOCHS        = 3
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5
MAX_LENGTH    = 128
LABEL2ID      = {"boilerplate": 0, "substantive": 1}
ID2LABEL      = {0: "boilerplate", 1: "substantive"}

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="stage5-finbert",
    job_type="train",
    tags=["stage:5-zoo", "purpose:train", "model:finbert", "split:oof"],
    config={
        "model": MODEL_NAME,
        "n_folds": N_FOLDS,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "seed": SEED,
    },
)
print(f"Run: {run.url}")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

art = run.use_artifact("colab-data:v0")
art_dir = Path(art.download())

train_df = pd.read_parquet(art_dir / "train_with_text.parquet")
val_df   = pd.read_parquet(art_dir / "val_with_text.parquet")
print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows")
print(f"Label distribution (train): {train_df['gold_label'].value_counts().to_dict()}")

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, recall_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Enable GPU via Runtime -> Change runtime type.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextDataset(TorchDataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=MAX_LENGTH, return_tensors="pt",
        )
        self.labels = labels

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


def make_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )


def get_probs(model, texts):
    """Batch inference -> p(substantive)."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            batch = list(texts.iloc[i : i + 64] if hasattr(texts, "iloc") else texts[i : i + 64])
            inputs = tokenizer(
                batch, truncation=True, padding=True,
                max_length=MAX_LENGTH, return_tensors="pt",
            ).to(DEVICE)
            logits = model(**inputs).logits
            probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
    return np.clip(np.array(all_probs, dtype=float), 0.0, 1.0)

In [ ]:
import time

y_train = (train_df["gold_label"] == "substantive").astype(int).values
groups  = (train_df["ticker"] + "_" + train_df["quarter"].astype(str)).values

skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs   = np.zeros(len(train_df), dtype=float)
fold_metrics = []

for fold_idx, (tr_idx, vl_idx) in enumerate(skf.split(train_df, y_train, groups)):
    print(f"\n=== Fold {fold_idx} ===")
    tr_ds    = TextDataset(train_df["text"].iloc[tr_idx], y_train[tr_idx])
    vl_texts = train_df["text"].iloc[vl_idx]
    vl_labels = y_train[vl_idx]

    model = make_model().to(DEVICE)

    args = TrainingArguments(
        output_dir=f"/content/fold_{fold_idx}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        evaluation_strategy="no",
        save_strategy="no",
        seed=SEED,
        fp16=(DEVICE == "cuda"),
        report_to="none",
        dataloader_pin_memory=False,
        logging_steps=50,
    )
    trainer = Trainer(model=model, args=args, train_dataset=tr_ds)
    trainer.train()

    probs = get_probs(model, vl_texts)
    oof_probs[vl_idx] = probs

    preds = (probs >= 0.5).astype(int)
    fm = {
        "fold": fold_idx,
        "macro_f1": float(f1_score(vl_labels, preds, average="macro")),
        "substantive_recall": float(recall_score(vl_labels, preds, pos_label=1)),
    }
    fold_metrics.append(fm)
    print(f"Fold {fold_idx}: F1={fm['macro_f1']:.4f}  recall={fm['substantive_recall']:.4f}")

    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

mean_f1     = float(np.mean([f["macro_f1"] for f in fold_metrics]))
mean_recall = float(np.mean([f["substantive_recall"] for f in fold_metrics]))
print(f"\nOOF macro-F1={mean_f1:.4f}  sub-recall={mean_recall:.4f}")

In [ ]:
y_val = (val_df["gold_label"] == "substantive").astype(int).values

MODEL_SAVE_DIR = "/content/finbert_model"
full_tr_ds = TextDataset(train_df["text"], y_train)

t0 = time.perf_counter()
final_model = make_model().to(DEVICE)
args = TrainingArguments(
    output_dir="/content/finbert_full",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    evaluation_strategy="no",
    save_strategy="no",
    seed=SEED,
    fp16=(DEVICE == "cuda"),
    report_to="none",
    dataloader_pin_memory=False,
)
trainer = Trainer(model=final_model, args=args, train_dataset=full_tr_ds)
trainer.train()
train_sec = time.perf_counter() - t0

t1 = time.perf_counter()
val_probs = get_probs(final_model, val_df["text"])
infer_sec = time.perf_counter() - t1
throughput = len(val_df) / infer_sec

val_preds  = (val_probs >= 0.5).astype(int)
val_f1     = float(f1_score(y_val, val_preds, average="macro"))
val_recall = float(recall_score(y_val, val_preds, pos_label=1))
print(f"Val F1={val_f1:.4f}  recall={val_recall:.4f}  throughput={throughput:.0f} sps")

final_model.save_pretrained(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f"Model saved to {MODEL_SAVE_DIR}")

oof_path      = Path("/content/oof_train_finbert.parquet")
val_pred_path = Path("/content/pred_val_finbert.parquet")
pd.DataFrame({"sentence_id": train_df["sentence_id"].values, "prob_substantive": oof_probs}).to_parquet(oof_path, index=False)
pd.DataFrame({"sentence_id": val_df["sentence_id"].values, "prob_substantive": val_probs}).to_parquet(val_pred_path, index=False)
print(f"OOF: {len(oof_probs)} rows | Val preds: {len(val_probs)} rows")

In [ ]:
wandb.log({
    "oof_macro_f1": mean_f1,
    "oof_substantive_recall": mean_recall,
    "val_macro_f1": val_f1,
    "val_substantive_recall": val_recall,
    "training_time_sec": train_sec,
    "infer_throughput_sps": throughput,
    **{f"fold_{f['fold']}_macro_f1": f["macro_f1"] for f in fold_metrics},
})

model_art = wandb.Artifact(
    "model-finbert",
    type="model",
    description="ProsusAI/finbert fine-tuned for boilerplate/substantive classification",
    metadata={"val_macro_f1": val_f1, "base_model": MODEL_NAME},
)
model_art.add_dir(MODEL_SAVE_DIR)
run.log_artifact(model_art)

oof_art = wandb.Artifact(
    "oof-finbert",
    type="predictions",
    description="5-fold OOF probs + val preds from FinBERT",
    metadata={"oof_macro_f1": mean_f1, "val_macro_f1": val_f1},
)
oof_art.add_file(str(oof_path), name="oof_train_finbert.parquet")
oof_art.add_file(str(val_pred_path), name="pred_val_finbert.parquet")
run.log_artifact(oof_art)

run.finish()
print("W&B run finished.")

In [ ]:
api = wandb.Api()
art = api.artifact(f"{WANDB_ENTITY}/{WANDB_PROJECT}/oof-finbert:v0")
files = [f.name for f in art.files()]
print(f"Confirmed oof-finbert:v0 files: {files}")
assert "oof_train_finbert.parquet" in files, "OOF parquet missing from artifact!"
assert "pred_val_finbert.parquet" in files, "Val-pred parquet missing from artifact!"
print("DONE — push to W&B successful")